# Analysis of DBN Model for Human Action Prediction

This notebook analyses the results of a Dynamic Bayesian Network (DBN) model for human action prediction.

The model is inspired by the study *The eye in hand: predicting others’ behaviour by integrating multiple sources of information*. The aim is to examine how gaze, hand preshape, and trajectory cues influence action prediction over time.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Load Data

In [ ]:
df = pd.read_csv("experiments/results.csv")
df.head()

## Check Data Structure

In [ ]:
df.info()
df["stage"].value_counts()

## Accuracy per Stage

In [ ]:
stage_accuracy = df.groupby("stage")["correct"].mean()
stage_accuracy = stage_accuracy.reindex(["gaze", "hand", "trajectory"])
stage_accuracy

Accuracy varies across stages, with later stages generally showing higher performance. This suggests that additional cues contribute to improved prediction.

In [ ]:
stage_accuracy.plot(kind="bar")
plt.title("Accuracy Across Stages")
plt.xlabel("Cue Stage")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

## Final Accuracy per Experiment

In [ ]:
final_stage = df[df["stage"] == "trajectory"]
final_accuracy = final_stage.groupby("experiment")["correct"].mean()
final_accuracy

Final accuracy differs across experimental conditions, particularly when early cues are misleading. This indicates that cue conflict affects prediction performance.

In [ ]:
final_accuracy.plot(kind="bar")
plt.title("Final DBN Prediction Accuracy Across Conditions")
plt.xlabel("Experiment Condition")
plt.ylabel("Final Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Belief Evolution

In [ ]:
exp = df[df["experiment"] == "gaze_wrong_hand_trajectory_correct_small"]

exp_mean = exp.groupby("stage")[["small_prob", "large_prob"]].mean()
exp_mean = exp_mean.reindex(["gaze", "hand", "trajectory"])

plt.plot(exp_mean.index, exp_mean["small_prob"], marker="o", label="Small")
plt.plot(exp_mean.index, exp_mean["large_prob"], marker="o", label="Large")
plt.legend()
plt.title("Belief Update Example")
plt.xlabel("Cue Stage")
plt.ylabel("Belief Probability")
plt.ylim(0, 1)
plt.show()

The belief changes across stages, showing that the model updates its prediction as new information becomes available. Early cues may bias the prediction, which is adjusted as stronger cues are processed.

## Effect of Noise

In [ ]:
if "noisy_observation" in df.columns:
    df["noisy_change"] = (df["observation"] != df["noisy_observation"]).astype(int)
    noise_rate = df["noisy_change"].mean()
    print("Overall noise rate:", noise_rate)

    noise_by_stage = df.groupby("stage")["noisy_change"].mean()
    noise_by_stage = noise_by_stage.reindex(["gaze", "hand", "trajectory"])
    display(noise_by_stage)
else:
    print("No noisy_observation column found. Noise tracking not enabled.")

Noise introduces variability, making the model more realistic compared to deterministic predictions.

In [ ]:
# Reliability Evolution
df.groupby("stage")[["gaze_reliability", "hand_reliability", "trajectory_reliability"]].mean()

Reliability values change over time, indicating learning from prediction outcomes.

## Discussion

The DBN model demonstrates that prediction improves as more informative cues become available. Early cues such as gaze provide weak information, while trajectory strongly determines the final prediction.

The introduction of noise results in non-perfect accuracy, aligning more closely with human performance. Additionally, the learning mechanism allows the model to adapt cue reliability over time.

However, the model remains simplified and does not capture full human variability or reaction time dynamics.